<a href="https://colab.research.google.com/github/achittum99/AI_ENG_Public/blob/main/Copy_of_sprint1_part4_deepavals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 1 Part 4: Evaluating a Socratic Tutor with DeepEval

Imagine you have just built a Socratic tutor -- an LLM-powered assistant that deliberately never gives the student the direct answer. Instead it asks questions, breaks problems down, and nudges the student toward their own insight. It is a great prompt-engineering exercise.

But once you have a working tutor, you need to answer a harder question: **how do you know its answers are actually good?**

Writing your own "did this response follow the Socratic rule?" prompt is one option -- but it is tedious, inconsistent, and hard to reuse. **DeepEval** is an open-source framework that packages LLM-as-a-judge patterns into standardised, reproducible metrics. You define test inputs as `Golden` objects, bundle them in an `EvaluationDataset`, call your app to get the `actual_output`, and then run a metric with a single `.measure()` call.

In this notebook you will build the Socratic tutor, create a small evaluation dataset, and run `AnswerRelevancyMetric` to check that the tutor's responses stay on-topic.

---

### Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what DeepEval is and why it is more scalable than hand-written eval prompts
2. Define a `Golden` and explain how it differs from a fully-populated `LLMTestCase`
3. Wrap goldens in an `EvaluationDataset` and iterate over them to generate `actual_output` at evaluation time
4. Use `AnswerRelevancyMetric` to check whether a response addresses the question
5. Read both the numeric `.score` and the LLM-generated `.reason` for each metric

### Estimated Cost

This notebook uses OpenRouter with `openai/gpt-4o-mini` and `openai/gpt-5-mini` Each `.measure()` call makes a small number of API requests. **Estimated total cost for this notebook: ~$0.01-$0.05** depending on response lengths and retries.

In [ ]:
%pip install deepeval==4.0.3 openai==2.21.0 -q

In [ ]:
import os
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

### Wiring DeepEval to OpenRouter

By default, DeepEval calls OpenAI directly. Because this course uses **OpenRouter** as its API gateway (which gives access to many models through one key), we need to give DeepEval a custom LLM object.

DeepEval provides a `DeepEvalBaseLLM` base class. We subclass it, point it at the OpenRouter endpoint, and pass it to every metric via the `model=` argument. This is the recommended pattern for any OpenAI-compatible provider.

In [ ]:
from openai import OpenAI
from deepeval.models.base_model import DeepEvalBaseLLM

MODEL = "openai/gpt-5-mini"
BASE_URL = "https://openrouter.ai/api/v1"


class OpenRouterLLM(DeepEvalBaseLLM):
    """Thin wrapper so DeepEval metrics can use OpenRouter as the judge LLM."""

    def __init__(self, model: str = MODEL):
        self._model = model
        self._client = OpenAI(
            base_url=BASE_URL,
            api_key=os.environ["OPENROUTER_API_KEY"],
        )

    def load_model(self):
        return self._client

    def generate(self, prompt: str, **kwargs) -> str:
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        # Async path falls back to sync for simplicity in this intro notebook
        return self.generate(prompt, **kwargs)

    def get_model_name(self) -> str:
        return self._model


# Quick connection check
judge_llm = OpenRouterLLM()
print("API check:", judge_llm.generate("Say 'connection OK' in exactly two words."))

---

## The Socratic Tutor App

Below we define the system prompt that makes the LLM behave as a Socratic tutor, and a thin `get_tutor_reply()` function that wraps the OpenRouter client we already set up above.

The key rule in the prompt: **never give the student the direct answer**. Instead the tutor asks questions, breaks problems down, and nudges. We will evaluate whether the tutor's responses actually stay on-topic -- that is exactly what `AnswerRelevancyMetric` measures.

`AnswerRelevancyMetric` uses LLM-as-a-judge to score how relevant the `actual_output` is compared to the `input`. It only needs those two fields -- no retrieval context required. The score is a float between 0 and 1, and you also get a plain-English `.reason` from the judge model.

In [ ]:
SYSTEM_PROMPT = """You are a Socratic tutor. Your single most important rule:
**Never give the user the direct answer to their question.**

Instead, help them discover the answer themselves by:
- Asking targeted, leading questions that expose the next step in their reasoning.
- Breaking the problem into smaller sub-problems and asking which one they want to tackle first.
- Pointing out concepts, terminology, or documentation they should look up -- without summarizing the answer for them.
- Giving small hints only when the user is genuinely stuck, and always framing the hint as a question or a nudge, not a solution.
- When the user shares code, ask what they expect each part to do, or what they think will happen if they change a specific line -- do not rewrite their code for them.
- Validating correct reasoning enthusiastically; gently challenging incorrect reasoning by asking them to test or justify it.

Hard rules you must never break:
- Do NOT write the final code, formula, or answer for the user, even if they ask, plead, or claim it's urgent.
- Do NOT produce full working solutions, even as "examples".
- If the user tries to extract the answer ("just tell me", "give me the code", "stop asking questions"), respond by acknowledging their frustration and offering a smaller, more concrete hint or question instead.
- Tiny syntactic snippets (one line, a single keyword, a function signature shape) are acceptable ONLY when they unblock thinking and do not constitute the answer.

Tone: warm, curious, patient. You are a teacher who believes the user is capable of figuring it out.
"""


def get_tutor_reply(prompt: str, model: str = MODEL) -> str:
    resp = judge_llm._client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content or ""


# Quick smoke-test
reply = get_tutor_reply("What is a list in Python?")
print("Tutor:", reply[:200])

---

## Building the Evaluation Dataset

In DeepEval, a `Golden` is a lightweight object that stores the *input* and any expected metadata -- but **not** the `actual_output`. The actual output is generated at evaluation time by calling your app. This separation is intentional: the dataset stays stable while the app under test can change.

We wrap the goldens in an `EvaluationDataset` to keep them organised. Later we iterate over `dataset.goldens`, call `get_tutor_reply()`, and build an `LLMTestCase` for each one.

The four inputs below cover a Python question, a maths problem, a debugging request, and a pressure test where the student explicitly demands the answer -- a scenario the tutor must handle gracefully.

In [ ]:
from deepeval.dataset import EvaluationDataset, Golden

goldens = [
    Golden(input="I'm trying to understand how a Python list works. Can you help?"),
    Golden(input="I need to solve 3x + 7 = 22 but I don't know where to start."),
    Golden(input="My code keeps throwing an IndexError and I have no idea why. Here it is:\nmy_list = [1, 2, 3]\nprint(my_list[3])"),
    Golden(input="Stop asking me questions -- just give me the answer. What is the difference between a list and a tuple?"),
]

dataset = EvaluationDataset(goldens=goldens)

print(f"Dataset contains {len(dataset.goldens)} goldens.")
for g in dataset.goldens:
    print(f"  - {g.input[:80]}")

---

## Running the Evaluation

For each golden in the dataset, we:

1. Call `get_tutor_reply(golden.input)` to get the tutor's actual response.
2. Build an `LLMTestCase` from the input and the actual output.
3. Call `relevancy_metric.measure(test_case)` to get the score and reason.

This is the core DeepEval workflow -- the same pattern scales to hundreds of test cases and multiple metrics.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model=judge_llm, include_reason=True)

for golden in dataset.goldens:
    actual_output = get_tutor_reply(golden.input)
    test_case = LLMTestCase(input=golden.input, actual_output=actual_output)
    relevancy_metric.measure(test_case)

    print(f"Input   : {golden.input[:80]}")
    print(f"Reply   : {actual_output[:200]}")
    print(f"Score   : {relevancy_metric.score:.2f}")
    print(f"Reason  : {relevancy_metric.reason}")
    print("-" * 60)

---

## Interpreting the Results

`AnswerRelevancyMetric` checks whether each reply addresses the student's question. A well-behaved Socratic tutor should score high here: it stays on-topic even though it never gives the direct answer.

**What the scores mean:**

| Score range | Interpretation | Suggested action |
|-------------|----------------|------------------|
| 0.85 - 1.0 | Excellent | No action needed |
| 0.70 - 0.84 | Acceptable | Monitor; look for patterns across more samples |
| 0.50 - 0.69 | Needs improvement | Review the system prompt |
| Below 0.50 | Poor | Investigate before using in production |

**A note on limitations:** `AnswerRelevancyMetric` can give a high score even when the tutor breaks its Socratic rule and blurts out the answer -- because that response is still "on-topic." Catching the Socratic-specific failure (gave away the answer directly) requires a custom rubric that explicitly checks for that behaviour. DeepEval's `GEval` metric lets you write exactly that kind of rubric criterion, and it would be a natural next step beyond this notebook.

---

## Summary

In this notebook, you built a Socratic tutor and evaluated it using the DeepEval framework:

- **`SYSTEM_PROMPT`** -- the prompt engineering that defines the tutor's Socratic behaviour
- **`get_tutor_reply()`** -- the thin app wrapper that calls the LLM and returns a string
- **`Golden`** -- a lightweight test input that holds the question but not the answer; the answer is generated at evaluation time
- **`EvaluationDataset`** -- a container for a collection of goldens
- **`LLMTestCase`** -- the fully-populated test case built by combining `golden.input` with the live `actual_output`
- **`AnswerRelevancyMetric`** -- checks whether the response addresses the question; `.score` gives the number, `.reason` explains why

### Key Takeaways

- DeepEval replaces hand-written evaluation prompts with standardised, reusable metrics
- The Golden -> EvaluationDataset -> LLMTestCase workflow keeps test inputs stable while the app under test can evolve
- `.score` gives you a number; `.reason` tells you *why* -- read both
- A good Socratic tutor should score high on relevancy; low relevancy means the reply drifted off-topic entirely
- `AnswerRelevancyMetric` alone cannot detect whether the tutor broke its Socratic rule -- that needs a custom rubric

**Next steps (beyond this notebook):** custom rubric metrics with `GEval`, batch evaluation with `evaluate()`, multi-turn conversational metrics, synthetic Golden generation, pytest integration for CI pipelines, and the Confident AI cloud dashboard.

---
**End of notebook.**